In [ ]:
import pandas as pd

# Carrega o dataset de coerência de processo
dataset_url = "https://huggingface.co/datasets/jotabpmn/bpmn-model-ia-generative/raw/main/process_coherence_dataset.csv"
coherence_df = pd.read_csv(dataset_url)

print("Dataset 'process_coherence_dataset.csv' carregado com sucesso!")
display(coherence_df.head())

Dataset 'process_coherence_dataset.csv' carregado com sucesso!


,process_name,text_description,bpmn_xml
0,Client Acquisition,After management indicates that new clients ar...,"<?xml version=""1.0"" encoding=""UTF-8""?>\n<bpmn:..."
1,Credit,The credit company receives the credit informa...,"<?xml version=""1.0"" encoding=""UTF-8""?>\n<bpmn:..."
2,Dispatching,"If goods shall be shipped, the secretary clari...","<?xml version=""1.0"" encoding=""UTF-8""?>\n<bpmn:..."
3,Hospital,The examination process can be summarised as f...,"<?xml version=""1.0"" encoding=""UTF-8""?>\n<bpmn2..."
4,Hotel,The Evanstonian is an upscale independent hote...,"<?xml version=""1.0"" encoding=""UTF-8""?>\n<bpmn:..."


In [ ]:
import re
import unicodedata
import xml.etree.ElementTree as ET
from xml.dom import minidom
import lxml.etree as ET_lxml
import networkx as nx # Adicionado para corrigir o erro NameError: name 'nx' is not definido
from sentence_transformers import SentenceTransformer, util # Mover import para o topo ou mantê-lo, mas garantir que 'util' esteja disponível
import spacy

def calculate_requirements_coverage_metric(reference_text, generated_xml, nlp_model, sentence_transformer_model):
    # Extrai o texto do modelo BPMN gerado
    texto_bpmn_gerado = extrair_texto_bpmn(generated_xml)

    # Gera embeddings para o texto de referência e o texto do BPMN gerado
    embeddings_texto_referencia = sentence_transformer_model.encode(reference_text, convert_to_tensor=True)
    embeddings_texto_bpmn_gerado = sentence_transformer_model.encode(texto_bpmn_gerado, convert_to_tensor=True)

    # Calcula a similaridade de cosseno
    similaridade_semantica = util.pytorch_cos_sim(embeddings_texto_referencia, embeddings_texto_bpmn_gerado).item() * 100
    return similaridade_semantica

def extrair_texto_bpmn(xml_string):
    # Extrai todos os nomes de elementos do BPMN para formar uma "descrição" do modelo
    root = ET_lxml.fromstring(xml_string.encode('utf-8'))
    nomes_elementos = []
    for elem in root.iter():
        if 'name' in elem.attrib:
            nomes_elementos.append(elem.attrib['name'])
    return ". ".join(nomes_elementos)


def calculate_ged_metric(generated_xml, reference_xml):
    # Converte os XMLs em grafos
    grafo_gerado_local = bpmn_to_networkx(generated_xml)
    grafo_referencia_local = bpmn_to_networkx(reference_xml)

    num_nodes_gerado = grafo_gerado_local.number_of_nodes()
    num_edges_gerado = grafo_gerado_local.number_of_edges()

    num_nodes_referencia = grafo_referencia_local.number_of_nodes()
    num_edges_referencia = grafo_referencia_local.number_of_edges()

    diff_nodes = abs(num_nodes_gerado - num_nodes_referencia)
    diff_edges = abs(num_edges_gerado - num_edges_referencia)

    ged_heuristico = diff_nodes + diff_edges

    total_elementos_gerado = num_nodes_gerado + num_edges_gerado
    total_elementos_referencia = num_nodes_referencia + num_edges_referencia

    if (total_elementos_gerado + total_elementos_referencia) > 0:
        max_possible_ged = total_elementos_gerado + total_elementos_referencia
        similaridade_ged = (1 - (ged_heuristico / max_possible_ged)) * 100
    else:
        similaridade_ged = 100.0 if ged_heuristico == 0 else 0.0

    return similaridade_ged

def validar_xml_sintaticamente(xml_string):
    try:
        # Tenta parsear o XML usando lxml para validação robusta
        # Deve-se codificar para bytes se a string contiver a declaração XML (e.g., <?xml ...?>)
        ET_lxml.fromstring(xml_string.encode('utf-8'))
        return 100.0 # XML é sintaticamente válido
    except ET_lxml.XMLSyntaxError:
        return 0.0 # XML não é sintaticamente válido

def run_syntactic_validation(xml_generated, xml_reference):
    validation_generated = validar_xml_sintaticamente(xml_generated)
    validation_reference = validar_xml_sintaticamente(xml_reference)
    return validation_generated, validation_reference




def sanitize_bpmn_id(value, prefix="Id"):
    value = str(value).strip()

    # Remove acentos
    value = unicodedata.normalize("NFKD", value)
    value = value.encode("ascii", "ignore").decode("ascii")

    # Mantém apenas letras, números e underscore
    value = re.sub(r"[^A-Za-z0-9_]", "_", value)

    # Compacta underscores repetidos
    value = re.sub(r"_+", "_", value).strip("_")

    # ID não deve começar com número
    if not value or not re.match(r"^[A-Za-z_]", value):
        value = f"{prefix}_{value}"

    return value


def make_unique_id(base_id, used_ids):
    candidate = base_id
    counter = 2

    while candidate in used_ids:
        candidate = f"{base_id}_{counter}"
        counter += 1

    used_ids.add(candidate)
    return candidate


def _initialize_bpmn_maps(process_data, used_ids):
    element_id_map = {}
    # print(f"DEBUG (_initialize_bpmn_maps): Conteúdo de process_data['elements'] no início da função: {json.dumps(process_data['elements'], indent=2)}")
    for element_data in process_data["elements"]:
        original_id = element_data["id"]
        clean_id = sanitize_bpmn_id(original_id, "Element")
        element_id_map[original_id] = make_unique_id(clean_id, used_ids)
    # print(f"DEBUG (_initialize_bpmn_maps): element_id_map após a construção: {element_id_map}")

    flow_id_map = {}
    for flow_data in process_data["sequence_flows"]:
        original_id = flow_data["id"]
        clean_id = sanitize_bpmn_id(original_id, "Flow")
        flow_id_map[original_id] = make_unique_id(clean_id, used_ids)
    return element_id_map, flow_id_map

def _create_bpmn_definitions_and_process(bpmn_namespace, xsi_namespace, process_name, used_ids):
    definitions_id = make_unique_id(
        sanitize_bpmn_id(f"Definitions_{process_name}", "Definitions"),
        used_ids,
    )
    process_id = make_unique_id(
        sanitize_bpmn_id(f"Process_{process_name}", "Process"),
        used_ids,
    )

    definitions = ET.Element(
        f"{{{bpmn_namespace}}}definitions",
        {
            "id": definitions_id,
            "targetNamespace": bpmn_namespace,
            f"{{{xsi_namespace}}}schemaLocation": f"{bpmn_namespace} BPMN20.xsd",
        },
    )

    process = ET.SubElement(
        definitions,
        f"{{{bpmn_namespace}}}process",
        {
            "id": process_id,
            "isExecutable": "false",
        },
    )
    return definitions, process, process_id

def _create_bpmn_elements_and_coords(process, bpmn_namespace, process_data, element_id_map, element_coords, x_offset, y_offset, gap, row_height):
    current_y = y_offset
    current_x = x_offset

    for element_data in process_data["elements"]:
        elem_type = element_data["type"]
        elem_id = element_id_map[element_data["id"]]
        elem_name = element_data.get("name", "")

        width, height = 0, 0

        if elem_type == "startEvent":
            ET.SubElement(
                process,
                f"{{{bpmn_namespace}}}startEvent",
                {"id": elem_id, "name": elem_name},
            )
            width, height = 36, 36

        elif elem_type == "endEvent":
            ET.SubElement(
                process,
                f"{{{bpmn_namespace}}}endEvent",
                {"id": elem_id, "name": elem_name},
            )
            width, height = 36, 36

        elif elem_type in ("userTask", "serviceTask"):
            ET.SubElement(
                process,
                f"{{{bpmn_namespace}}}{elem_type}",
                {"id": elem_id, "name": elem_name},
            )
            width, height = 100, 80

        elif elem_type == "exclusiveGateway":
            ET.SubElement(
                process,
                f"{{{bpmn_namespace}}}exclusiveGateway",
                {
                    "id": elem_id,
                    "name": elem_name,
                    "gatewayDirection": "Diverging",
                },
            )
            width, height = 50, 50

        else:
            continue

        element_coords[elem_id] = {
            "x": current_x,
            "y": current_y,
            "width": width,
            "height": height,
        }

        current_x += width + gap

        if current_x > 800: # Simple layout logic, adjust as needed
            current_x = x_offset
            current_y += row_height + gap
    return element_coords

def _create_bpmn_sequence_flows(process, bpmn_namespace, process_data, element_id_map, flow_id_map):
    for flow_data in process_data["sequence_flows"]:
        source_original_id = flow_data["from"]
        target_original_id = flow_data["to"]

        if source_original_id not in element_id_map:
            raise ValueError(
                f"Fluxo '{flow_data['id']}' aponta para origem inexistente: {source_original_id}"
            )

        if target_original_id not in element_id_map:
            raise ValueError(
                f"Fluxo '{flow_data['id']}' aponta para destino inexistente: {target_original_id}"
            )

        ET.SubElement(
            process,
            f"{{{bpmn_namespace}}}sequenceFlow",
            {
                "id": flow_id_map[flow_data["id"]],
                "sourceRef": element_id_map[source_original_id],
                "targetRef": element_id_map[target_original_id],
            },
        )

def _create_bpmndi_diagram_and_plane(definitions, process_id, bpmndi_namespace):
    bpmndi_diagram = ET.SubElement(
        definitions,
        f"{{{bpmndi_namespace}}}BPMNDiagram",
        {"id": "BPMNDiagram_1"},
    )

    bpmndi_plane = ET.SubElement(
        bpmndi_diagram,
        f"{{{bpmndi_namespace}}}BPMNPlane",
        {
            "id": "BPMNPlane_1",
            "bpmnElement": process_id,
        },
    )
    return bpmndi_plane

def _create_bpmndi_shapes(bpmndi_plane, bpmndi_namespace, omgdc_namespace, process_data, element_id_map, element_coords, used_ids):
    for element_data in process_data["elements"]:
        elem_id = element_id_map[element_data["id"]]
        coords = element_coords.get(elem_id)

        if not coords:
            continue

        shape_id = make_unique_id(
            sanitize_bpmn_id(f"BPMNShape_{elem_id}", "BPMNShape"),
            used_ids,
        )

        shape = ET.SubElement(
            bpmndi_plane,
            f"{{{bpmndi_namespace}}}BPMNShape",
            {
                "id": shape_id,
                "bpmnElement": elem_id,
            },
        )

        ET.SubElement(
            shape,
            f"{{{omgdc_namespace}}}Bounds",
            {
                "x": str(coords["x"]),
                "y": str(coords["y"]),
                "width": str(coords["width"]),
                "height": str(coords["height"]),
            },
        )

def _create_bpmndi_edges(bpmndi_plane, bpmndi_namespace, omgdi_namespace, process_data, element_id_map, flow_id_map, element_coords, used_ids):
    for flow_data in process_data["sequence_flows"]:
        flow_id = flow_id_map[flow_data["id"]]
        source_id = element_id_map[flow_data["from"]]
        target_id = element_id_map[flow_data["to"]]

        source_coords = element_coords.get(source_id)
        target_coords = element_coords.get(target_id)

        if not source_coords or not target_coords:
            continue

        edge_id = make_unique_id(
            sanitize_bpmn_id(f"BPMNEdge_{flow_id}", "BPMNEdge"),
            used_ids,
        )

        edge = ET.SubElement(
            bpmndi_plane,
            f"{{{bpmndi_namespace}}}BPMNEdge",
            {
                "id": edge_id,
                "bpmnElement": flow_id,
            },
        )

        ET.SubElement(
            edge,
            f"{{{omgdi_namespace}}}Waypoint",
            {
                "x": str(source_coords["x"] + source_coords["width"]),
                "y": str(source_coords["y"] + source_coords["height"] / 2),
            },
        )

        ET.SubElement(
            edge,
            f"{{{omgdi_namespace}}}Waypoint",
            {
                "x": str(target_coords["x"]),
                "y": str(target_coords["y"] + target_coords["height"] / 2),
            },
        )

def _format_and_save_xml(definitions, filename="processo_gerado.bpmn"):
    rough_string = ET.tostring(definitions, "utf-8")
    reparsed = minidom.parseString(rough_string)
    pretty_xml_as_string = reparsed.toprettyxml(indent="  ")

    pretty_xml_as_string = "\n".join(pretty_xml_as_string.split("\n")[1:])
    pretty_xml_as_string = "<?xml version=\"1.0\" encoding=\"UTF-8\"?>\n" + pretty_xml_as_string

    return pretty_xml_as_string


def gerar_bpmn_definitivo(process_data):
    bpmn_namespace = "http://www.omg.org/spec/BPMN/20100524/MODEL"
    xsi_namespace = "http://www.w3.org/2001/XMLSchema-instance"
    bpmndi_namespace = "http://www.omg.org/spec/BPMN/20100524/DI"
    omgdc_namespace = "http://www.omg.org/spec/DD/20100524/DC"
    omgdi_namespace = "http://www.omg.org/spec/DD/20100524/DI"

    ET.register_namespace("bpmn", bpmn_namespace)
    ET.register_namespace("xsi", xsi_namespace)
    ET.register_namespace("bpmndi", bpmndi_namespace)
    ET.register_namespace("omgdc", omgdc_namespace)
    ET.register_namespace("omgdi", omgdi_namespace)

    used_ids = set()
    process_name = process_data["process_name"]

    # 1. Initialize BPMN elements and flows maps
    element_id_map, flow_id_map = _initialize_bpmn_maps(process_data, used_ids)

    # 2. Create BPMN Definitions and Process
    definitions, process, process_id = _create_bpmn_definitions_and_process(
        bpmn_namespace, xsi_namespace, process_name, used_ids
    )

    # 3. Create BPMN Elements and calculate coordinates
    element_coords = {}
    x_offset = 100
    y_offset = 100
    gap = 150
    row_height = 80
    element_coords = _create_bpmn_elements_and_coords(
        process, bpmn_namespace, process_data, element_id_map, element_coords,
        x_offset, y_offset, gap, row_height
    )

    # 4. Create BPMN Sequence Flows
    _create_bpmn_sequence_flows(
        process, bpmn_namespace, process_data, element_id_map, flow_id_map
    )

    # 5. Create BPMN Diagram and Plane
    bpmndi_plane = _create_bpmndi_diagram_and_plane(
        definitions, process_id, bpmndi_namespace
    )

    # 6. Create BPMN Shapes
    _create_bpmndi_shapes(
        bpmndi_plane, bpmndi_namespace, omgdc_namespace,
        process_data, element_id_map, element_coords, used_ids
    )

    # 7. Create BPMN Edges
    _create_bpmndi_edges(
        bpmndi_plane, bpmndi_namespace, omgdi_namespace,
        process_data, element_id_map, flow_id_map, element_coords, used_ids
    )

    # 8. Format and return XML
    pretty_xml_as_string = _format_and_save_xml(definitions)

    print(f"XML BPMN gerado com IDs compatíveis. Retornando string...")

    return pretty_xml_as_string

def calculate_structural_complexity_metric(generated_xml):
    # Converte o XML gerado em grafo
    grafo_gerado_local = bpmn_to_networkx(generated_xml)

    # Calcula a densidade do grafo
    # A densidade de um grafo direcionado é: E / (V * (V-1))
    num_nodes_local = grafo_gerado_local.number_of_nodes()
    num_edges_local = grafo_gerado_local.number_of_edges()

    if num_nodes_local > 1:
        densidade_grafo = num_edges_local / (num_nodes_local * (num_nodes_local - 1))
    else:
        densidade_grafo = 0.0

    return densidade_grafo * 100

def bpmn_to_networkx(xml_string):
    graph = nx.DiGraph()
    root = ET_lxml.fromstring(xml_string.encode('utf-8'))

    # List of BPMN element tags to consider as nodes
    bpmn_node_tags = [
        '{http://www.omg.org/spec/BPMN/20100524/MODEL}task',
        '{http://www.omg.org/spec/BPMN/20100524/MODEL}startEvent',
        '{http://www.omg.org/spec/BPMN/20100524/MODEL}endEvent',
        '{http://www.omg.org/spec/BPMN/20100524/MODEL}exclusiveGateway',
        '{http://www.omg.org/spec/BPMN/20100524/MODEL}userTask',
        '{http://www.omg.org/spec/BPMN/20100524/MODEL}serviceTask'
    ]

    # Add nodes
    for tag in bpmn_node_tags:
        for elem in root.iter(tag):
            graph.add_node(elem.attrib['id'], type=elem.tag.split('}')[-1], name=elem.attrib.get('name', ''))

    # Add edges
    for flow in root.iter('{http://www.omg.org/spec/BPMN/20100524/MODEL}sequenceFlow'):
        source_ref = flow.attrib.get('sourceRef')
        target_ref = flow.attrib.get('targetRef')
        if source_ref and target_ref:
            graph.add_edge(source_ref, target_ref)
    return graph


def validar_json_sintaticamente(json_string):
    """
    Tenta parsear a string JSON para validar sua sintaxe.
    Retorna True se o JSON for sintaticamente válido, False caso contrário.
    """
    try:
        json.loads(json_string)
        return True
    except json.JSONDecodeError:
        return False

def validar_bpmn_logicamente(process_data):
    """
    Valida a lógica BPMN de um dicionário de processo, verificando se os IDs de origem e destino
    dos sequence_flows existem na lista de elementos.
    Retorna True se for logicamente válido, False caso contrário.
    """
    elements_ids = {elem["id"] for elem in process_data.get("elements", [])}

    for flow_data in process_data.get("sequence_flows", []):
        source_id = flow_data.get("from")
        target_id = flow_data.get("to")

        if source_id not in elements_ids:
            print(f"❌ Erro de validação lógica BPMN: Fluxo '{flow_data.get('id', 'N/A')}' aponta para origem inexistente: '{source_id}'")
            return False
        if target_id not in elements_ids:
            print(f"❌ Erro de validação lógica BPMN: Fluxo '{flow_data.get('id', 'N/A')}' aponta para destino inexistente: '{target_id}'")
            return False
    return True



def run_model_until_valid_json(
    model, # Gemini model object
    prompt_sistema, # System prompt for Gemini
    sua_nova_descricao, # User description for Gemini
    generation_config, # Generation config for Gemini
    max_attempts=5
):

    retry_count = 0
    while retry_count < max_attempts:
        retry_count += 1
        print(f"\n⚡ Tentativa {retry_count} de {max_attempts}: Gerando texto com Gemini...")

        # 1. Gerar o texto usando a API do Gemini
        try:
            response = model.generate_content(
                [prompt_sistema, f"Converta este processo:\n{sua_nova_descricao}"],
                generation_config=generation_config
            )
            cleaned_output = response.text.strip()
        except Exception as e:
            print(f"❌ Erro ao chamar a API do Gemini na tentativa {retry_count}: {e}")
            # Se a API falhar, não há JSON para validar, então continuamos para a próxima tentativa
            continue

        # 2. Extrair o bloco JSON da string gerada (lógica existente)
        codigo_json = ""
        json_start_marker = "```json"
        json_end_marker = "```"

        start_idx = cleaned_output.find(json_start_marker)
        end_idx = cleaned_output.find(json_end_marker, start_idx + len(json_start_marker))

        if start_idx != -1 and end_idx != -1:
            codigo_json = cleaned_output[start_idx + len(json_start_marker):end_idx].strip()
        else:
            print("Warning: Bloco Markdown JSON não encontrado. Tentando chaves estruturais.")
            first_brace = cleaned_output.find('{')
            last_brace = cleaned_output.rfind('}')
            if first_brace != -1 and last_brace != -1 and last_brace > first_brace:
                codigo_json = cleaned_output[first_brace:last_brace+1].strip()

        # 3. Validar a sintaxe JSON (lógica existente)
        if validar_json_sintaticamente(codigo_json):
            print(f"✅ JSON sintaticamente válido encontrado na tentativa {retry_count}. Tentando validação lógica...")
            try:
                dados_processo = json.loads(codigo_json)

                # NOVA VALIDAÇÃO: Validação lógica do BPMN (consistência de flows e elementos) (lógica existente)
                if validar_bpmn_logicamente(dados_processo):
                    print(f"✅ JSON sintaticamente e logicamente válido encontrado na tentativa {retry_count}. Retornando.\n")
                    return dados_processo, retry_count
                else:
                    print(f"❌ JSON sintaticamente válido, mas falhou na validação lógica BPMN na tentativa {retry_count}. Retentando...")
            except json.JSONDecodeError as e:
                print(f"Erro inesperado ao parsear JSON válido: {e}")
        else:
            print(f"❌ JSON inválido na tentativa {retry_count}. Conteúdo bruto: {cleaned_output[:500]}...")

    print(f"\n⛔ Não foi possível gerar um JSON sintaticamente e logicamente válido após {max_attempts} tentativas.")
    return None, max_attempts

In [ ]:
import os
import json
import warnings
# Remove a importação da OpenAI, pois não será mais usada para o Gemini
# from openai import OpenAI
from google.colab import userdata
import google.generativeai as genai

# Silencia avisos desnecessários
warnings.filterwarnings("ignore")

# 1. Configuração da API do Gemini
# Certifique-se de ter sua chave de API do Gemini salva nos segredos do Colab como 'GOOGLE_API_KEY'
# É recomendável usar o recurso de segredos do Colab para armazenar chaves de API com segurança.

genai.configure(api_key="AQ.Ab8RN6J1VBO8xt9B-EbB3-bVO7c50MJtxWnEcw8UmdjKyHntHg")

# Inicializa o modelo Gemini (vamos tentar listar os modelos disponíveis para ter certeza)
# model = genai.GenerativeModel('gemini-1.5-flash') # Mantendo 'gemini-1.5-flash' por enquanto, mas verificaremos a disponibilidade

# ----- CÓDIGO PARA LISTAR MODELOS DISPONÍVEIS -----
print("Verificando modelos Gemini disponíveis que suportam 'generateContent'...")

# Escolhe um modelo. Priorizamos os modelos 'flash' mais recentes e depois o 'pro'.
selected_model_name = "models/gemini-flash-latest"
model = genai.GenerativeModel(selected_model_name)


# Captura o primeiro registro do dataset coherence_df
sua_nova_descricao = coherence_df.loc[0, 'text_description']
print(f"Nova descrição do processo carregada do dataset:\n{sua_nova_descricao}\n")

prompt_sistema = """You are a strict BPMN translator. Your sole function is to map the user's text into the exact JSON schema provided below.

COMPLIANCE RULES:
1. The 'elements' list MUST contain only physical nodes (startEvent, userTask, serviceTask, exclusiveGateway, endEvent). Never place 'sequenceFlow' here.
2. The 'sequence_flows' list MUST map all connections using EXACTLY the keys 'id', 'from', and 'to'. Never use 'source' or 'target'.
3. Ensure that EVERY created element has at least one connection linked to it (avoid orphaned nodes).
4. GATEWAY RULE: If there is an 'exclusiveGateway' (decision point), make sure that the 'sequence_flows' list contains ALL paths described in the text (e.g., if there are two options like "simple" and "complex", the gateway MUST have two separate outgoing flows ('from') pointing to their respective destinations).
5. MANDATORY CLOSURE RULE: Every process needs to have a clear start (startEvent) and a clear end (endEvent). Ensure that the 'elements' list contains one or more 'endEvent' nodes, and that all tasks concluding the process create a 'sequence_flow' pointing to that 'endEvent'.

Return ONLY the structured JSON below, with no additional text, explanations, or markdown:
{
  "process_name": "Process_Name",
  "elements": [
    {"id": "node_id", "type": "node_type", "name": "Friendly name"}
  ],
  "sequence_flows": [
    {"id": "flow_id", "from": "source_id", "to": "destination_id"}
  ]
}"""


print("⚡ IA processando via API do Gemini...")



Verificando modelos Gemini disponíveis que suportam 'generateContent'...
Nova descrição do processo carregada do dataset:
After management indicates that new clients are needed, the marketing team will perform a market analysis and prepare a portfolio presentation.
When they have sent out offers to potential clients the marketing team informs management about the responses.
Management then starts negotiations by sending out a detailed offer to the client.
If the client accepts the offer, management will prepare a contract that will be signed.
Otherwise, management analyzes the reason why their offer failed in order to improve in the future.

⚡ IA processando via API do Gemini...


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


### Função para Gerar XML BPMN 2.0 a partir do JSON

A função `gerar_bpmn_definitivo` recebe um dicionário Python (como `dados_processo`) e constrói um arquivo XML BPMN 2.0 completo. Ela inclui a lógica para criar todos os elementos do processo (eventos, tarefas, gateways) e suas conexões (`sequenceFlows`), além de gerar automaticamente a representação visual (`BPMNDiagram`) com coordenadas para cada elemento, garantindo um layout básico e funcional do seu processo.

In [ ]:
import re
import unicodedata
import xml.etree.ElementTree as ET
from xml.dom import minidom
import lxml.etree as ET_lxml
import networkx as nx
import json # Added for debugging and JSON handling
import time # Added for timing JSON generation

def calculate_requirements_coverage_metric(reference_text, generated_xml, nlp_model, sentence_transformer_model):
    # Extrai o texto do modelo BPMN gerado
    texto_bpmn_gerado = extrair_texto_bpmn(generated_xml)

    # Gera embeddings para o texto de referência e o texto do BPMN gerado
    embeddings_texto_referencia = sentence_transformer_model.encode(reference_text, convert_to_tensor=True)
    embeddings_texto_bpmn_gerado = sentence_transformer_model.encode(texto_bpmn_gerado, convert_to_tensor=True)

    # Calcula a similaridade de cosseno
    similaridade_semantica = util.pytorch_cos_sim(embeddings_texto_referencia, embeddings_texto_bpmn_gerado).item() * 100
    return similaridade_semantica

def extrair_texto_bpmn(xml_string):
    # Extrai todos os nomes de elementos do BPMN para formar uma "descrição" do modelo
    root = ET_lxml.fromstring(xml_string.encode('utf-8'))
    nomes_elementos = []
    for elem in root.iter():
        if 'name' in elem.attrib:
            nomes_elementos.append(elem.attrib['name'])
    return ". ".join(nomes_elementos)


def calculate_ged_metric(generated_xml, reference_xml):
    # Converte os XMLs em grafos
    grafo_gerado_local = bpmn_to_networkx(generated_xml)
    grafo_referencia_local = bpmn_to_networkx(reference_xml)

    num_nodes_gerado = grafo_gerado_local.number_of_nodes()
    num_edges_gerado = grafo_gerado_local.number_of_edges()

    num_nodes_referencia = grafo_referencia_local.number_of_nodes()
    num_edges_referencia = grafo_referencia_local.number_of_edges()

    diff_nodes = abs(num_nodes_gerado - num_nodes_referencia)
    diff_edges = abs(num_edges_gerado - num_edges_referencia)

    ged_heuristico = diff_nodes + diff_edges

    total_elementos_gerado = num_nodes_gerado + num_edges_gerado
    total_elementos_referencia = num_nodes_referencia + num_edges_referencia

    if (total_elementos_gerado + total_elementos_referencia) > 0:
        max_possible_ged = total_elementos_gerado + total_elementos_referencia
        similaridade_ged = (1 - (ged_heuristico / max_possible_ged)) * 100
    else:
        similaridade_ged = 100.0 if ged_heuristico == 0 else 0.0

    return similaridade_ged

def validar_xml_sintaticamente(xml_string):
    try:
        # Tenta parsear o XML usando lxml para validação robusta
        # Deve-se codificar para bytes se a string contiver a declaração XML (e.g., <?xml ...?>)
        ET_lxml.fromstring(xml_string.encode('utf-8'))
        return 100.0 # XML é sintaticamente válido
    except ET_lxml.XMLSyntaxError:
        return 0.0 # XML não é sintaticamente válido

def run_syntactic_validation(xml_generated, xml_reference):
    validation_generated = validar_xml_sintaticamente(xml_generated)
    validation_reference = validar_xml_sintaticamente(xml_reference)
    return validation_generated, validation_reference

def sanitize_bpmn_id(value, prefix="Id"):
    value = str(value).strip()

    # Remove acentos
    value = unicodedata.normalize("NFKD", value)
    value = value.encode("ascii", "ignore").decode("ascii")

    # Mantém apenas letras, números e underscore
    value = re.sub(r"[^A-Za-z0-9_]", "_", value)

    # Compacta underscores repetidos
    value = re.sub(r"_+", "_", value).strip("_")

    # ID não deve começar com número
    if not value or not re.match(r"^[A-Za-z_]", value):
        value = f"{prefix}_{value}"

    return value


def make_unique_id(base_id, used_ids):
    candidate = base_id
    counter = 2

    while candidate in used_ids:
        candidate = f"{base_id}_{counter}"
        counter += 1

    used_ids.add(candidate)
    return candidate

# --- Modularized Helper Functions ---
def _initialize_bpmn_maps(process_data, used_ids):
    element_id_map = {}
    print(f"DEBUG (_initialize_bpmn_maps): Conteúdo de process_data['elements'] no início da função: {json.dumps(process_data['elements'], indent=2)}")
    for element_data in process_data["elements"]:
        original_id = element_data["id"]
        clean_id = sanitize_bpmn_id(original_id, "Element")
        element_id_map[original_id] = make_unique_id(clean_id, used_ids)
    print(f"DEBUG (_initialize_bpmn_maps): element_id_map após a construção: {element_id_map}")

    flow_id_map = {}
    for flow_data in process_data["sequence_flows"]:
        original_id = flow_data["id"]
        clean_id = sanitize_bpmn_id(original_id, "Flow")
        flow_id_map[original_id] = make_unique_id(clean_id, used_ids)
    return element_id_map, flow_id_map

def _create_bpmn_definitions_and_process(bpmn_namespace, xsi_namespace, process_name, used_ids):
    definitions_id = make_unique_id(
        sanitize_bpmn_id(f"Definitions_{process_name}", "Definitions"),
        used_ids,
    )
    process_id = make_unique_id(
        sanitize_bpmn_id(f"Process_{process_name}", "Process"),
        used_ids,
    )

    definitions = ET.Element(
        f"{{{bpmn_namespace}}}definitions",
        {
            "id": definitions_id,
            "targetNamespace": bpmn_namespace,
            f"{{{xsi_namespace}}}schemaLocation": f"{bpmn_namespace} BPMN20.xsd",
        },
    )

    process = ET.SubElement(
        definitions,
        f"{{{bpmn_namespace}}}process",
        {
            "id": process_id,
            "isExecutable": "false",
        },
    )
    return definitions, process, process_id

def _create_bpmn_elements_and_coords(process, bpmn_namespace, process_data, element_id_map, element_coords, x_offset, y_offset, gap, row_height):
    current_y = y_offset
    current_x = x_offset

    for element_data in process_data["elements"]:
        elem_type = element_data["type"]
        elem_id = element_id_map[element_data["id"]]
        elem_name = element_data.get("name", "")

        width, height = 0, 0

        if elem_type == "startEvent":
            ET.SubElement(
                process,
                f"{{{bpmn_namespace}}}startEvent",
                {"id": elem_id, "name": elem_name},
            )
            width, height = 36, 36

        elif elem_type == "endEvent":
            ET.SubElement(
                process,
                f"{{{bpmn_namespace}}}endEvent",
                {"id": elem_id, "name": elem_name},
            )
            width, height = 36, 36

        elif elem_type in ("userTask", "serviceTask"):
            ET.SubElement(
                process,
                f"{{{bpmn_namespace}}}{elem_type}",
                {"id": elem_id, "name": elem_name},
            )
            width, height = 100, 80

        elif elem_type == "exclusiveGateway":
            ET.SubElement(
                process,
                f"{{{bpmn_namespace}}}exclusiveGateway",
                {
                    "id": elem_id,
                    "name": elem_name,
                    "gatewayDirection": "Diverging",
                },
            )
            width, height = 50, 50

        else:
            continue

        element_coords[elem_id] = {
            "x": current_x,
            "y": current_y,
            "width": width,
            "height": height,
        }

        current_x += width + gap

        if current_x > 800: # Simple layout logic, adjust as needed
            current_x = x_offset
            current_y += row_height + gap
    return element_coords

def _create_bpmn_sequence_flows(process, bpmn_namespace, process_data, element_id_map, flow_id_map):
    for flow_data in process_data["sequence_flows"]:
        source_original_id = flow_data["from"]
        target_original_id = flow_data["to"]

        if source_original_id not in element_id_map:
            raise ValueError(
                f"Fluxo '{flow_data['id']}' aponta para origem inexistente: {source_original_id}"
            )

        if target_original_id not in element_id_map:
            raise ValueError(
                f"Fluxo '{flow_data['id']}' aponta para destino inexistente: {target_original_id}"
            )

        ET.SubElement(
            process,
            f"{{{bpmn_namespace}}}sequenceFlow",
            {
                "id": flow_id_map[flow_data["id"]],
                "sourceRef": element_id_map[source_original_id],
                "targetRef": element_id_map[target_original_id],
            },
        )

def _create_bpmndi_diagram_and_plane(definitions, process_id, bpmndi_namespace):
    bpmndi_diagram = ET.SubElement(
        definitions,
        f"{{{bpmndi_namespace}}}BPMNDiagram",
        {"id": "BPMNDiagram_1"},
    )

    bpmndi_plane = ET.SubElement(
        bpmndi_diagram,
        f"{{{bpmndi_namespace}}}BPMNPlane",
        {
            "id": "BPMNPlane_1",
            "bpmnElement": process_id,
        },
    )
    return bpmndi_plane

def _create_bpmndi_shapes(bpmndi_plane, bpmndi_namespace, omgdc_namespace, process_data, element_id_map, element_coords, used_ids):
    for element_data in process_data["elements"]:
        elem_id = element_id_map[element_data["id"]]
        coords = element_coords.get(elem_id)

        if not coords:
            continue

        shape_id = make_unique_id(
            sanitize_bpmn_id(f"BPMNShape_{elem_id}", "BPMNShape"),
            used_ids,
        )

        shape = ET.SubElement(
            bpmndi_plane,
            f"{{{bpmndi_namespace}}}BPMNShape",
            {
                "id": shape_id,
                "bpmnElement": elem_id,
            },
        )

        ET.SubElement(
            shape,
            f"{{{omgdc_namespace}}}Bounds",
            {
                "x": str(coords["x"]),
                "y": str(coords["y"]),
                "width": str(coords["width"]),
                "height": str(coords["height"]),
            },
        )

def _create_bpmndi_edges(bpmndi_plane, bpmndi_namespace, omgdi_namespace, process_data, element_id_map, flow_id_map, element_coords, used_ids):
    for flow_data in process_data["sequence_flows"]:
        flow_id = flow_id_map[flow_data["id"]]
        source_id = element_id_map[flow_data["from"]]
        target_id = element_id_map[flow_data["to"]]

        source_coords = element_coords.get(source_id)
        target_coords = element_coords.get(target_id)

        if not source_coords or not target_coords:
            continue

        edge_id = make_unique_id(
            sanitize_bpmn_id(f"BPMNEdge_{flow_id}", "BPMNEdge"),
            used_ids,
        )

        edge = ET.SubElement(
            bpmndi_plane,
            f"{{{bpmndi_namespace}}}BPMNEdge",
            {
                "id": edge_id,
                "bpmnElement": flow_id,
            },
        )

        ET.SubElement(
            edge,
            f"{{{omgdi_namespace}}}Waypoint",
            {
                "x": str(source_coords["x"] + source_coords["width"]),
                "y": str(source_coords["y"] + source_coords["height"] / 2),
            },
        )

        ET.SubElement(
            edge,
            f"{{{omgdi_namespace}}}Waypoint",
            {
                "x": str(target_coords["x"]),
                "y": str(target_coords["y"] + target_coords["height"] / 2),
            },
        )

def _format_and_save_xml(definitions, filename="processo_gerado.bpmn"):
    rough_string = ET.tostring(definitions, "utf-8")
    reparsed = minidom.parseString(rough_string)
    pretty_xml_as_string = reparsed.toprettyxml(indent="  ")

    pretty_xml_as_string = "\n".join(pretty_xml_as_string.split("\n")[1:])
    pretty_xml_as_string = "<?xml version=\"1.0\" encoding=\"UTF-8\"?>\n" + pretty_xml_as_string

    # The previous version saved the file here, but the function's return was `pretty_xml_as_string`
    # If the user's intent is to just get the string, we should remove file saving.
    # For now, keeping the file saving but commenting it out to align with previous return.
    # with open(filename, "w", encoding="utf-8") as f:
    #     f.write(pretty_xml_as_string)
    # print(f"Arquivo final '{filename}' gerado com IDs compatíveis com BPMN.io.")

    return pretty_xml_as_string


def gerar_bpmn_definitivo(process_data):
    bpmn_namespace = "http://www.omg.org/spec/BPMN/20100524/MODEL"
    xsi_namespace = "http://www.w3.org/2001/XMLSchema-instance"
    bpmndi_namespace = "http://www.omg.org/spec/BPMN/20100524/DI"
    omgdc_namespace = "http://www.omg.org/spec/DD/20100524/DC"
    omgdi_namespace = "http://www.omg.org/spec/DD/20100524/DI"

    ET.register_namespace("bpmn", bpmn_namespace)
    ET.register_namespace("xsi", xsi_namespace)
    ET.register_namespace("bpmndi", bpmndi_namespace)
    ET.register_namespace("omgdc", omgdc_namespace)
    ET.register_namespace("omgdi", omgdi_namespace)

    used_ids = set()
    process_name = process_data["process_name"]

    # 1. Initialize BPMN elements and flows maps
    element_id_map, flow_id_map = _initialize_bpmn_maps(process_data, used_ids)

    # 2. Create BPMN Definitions and Process
    definitions, process, process_id = _create_bpmn_definitions_and_process(
        bpmn_namespace, xsi_namespace, process_name, used_ids
    )

    # 3. Create BPMN Elements and calculate coordinates
    element_coords = {}
    x_offset = 100
    y_offset = 100
    gap = 150
    row_height = 80
    element_coords = _create_bpmn_elements_and_coords(
        process, bpmn_namespace, process_data, element_id_map, element_coords,
        x_offset, y_offset, gap, row_height
    )

    # 4. Create BPMN Sequence Flows
    _create_bpmn_sequence_flows(
        process, bpmn_namespace, process_data, element_id_map, flow_id_map
    )

    # 5. Create BPMN Diagram and Plane
    bpmndi_plane = _create_bpmndi_diagram_and_plane(
        definitions, process_id, bpmndi_namespace
    )

    # 6. Create BPMN Shapes
    _create_bpmndi_shapes(
        bpmndi_plane, bpmndi_namespace, omgdc_namespace,
        process_data, element_id_map, element_coords, used_ids
    )

    # 7. Create BPMN Edges
    _create_bpmndi_edges(
        bpmndi_plane, bpmndi_namespace, omgdi_namespace,
        process_data, element_id_map, flow_id_map, element_coords, used_ids
    )

    # 8. Format and return XML
    pretty_xml_as_string = _format_and_save_xml(definitions)

    print(f"XML BPMN gerado com IDs compatíveis. Retornando string...")

    # The previous version tried to download the file, but the function returned a string.
    # For now, I will keep the download logic commented to align with returning the string.
    # try:
    #     files.download(filename)
    # except NameError:
    #     pass

    return pretty_xml_as_string

def calculate_structural_complexity_metric(generated_xml):
    # Converte o XML gerado em grafo
    grafo_gerado_local = bpmn_to_networkx(generated_xml)

    # Calcula a densidade do grafo
    # A densidade de um grafo direcionado é: E / (V * (V-1))
    num_nodes_local = grafo_gerado_local.number_of_nodes()
    num_edges_local = grafo_gerado_local.number_of_edges()

    if num_nodes_local > 1:
        densidade_grafo = num_edges_local / (num_nodes_local * (num_nodes_local - 1))
    else:
        densidade_grafo = 0.0

    return densidade_grafo * 100

def bpmn_to_networkx(xml_string):
    graph = nx.DiGraph()
    root = ET_lxml.fromstring(xml_string.encode('utf-8'))

    # List of BPMN element tags to consider as nodes
    bpmn_node_tags = [
        '{http://www.omg.org/spec/BPMN/20100524/MODEL}task',
        '{http://www.omg.org/spec/BPMN/20100524/MODEL}startEvent',
        '{http://www.omg.org/spec/BPMN/20100524/MODEL}endEvent',
        '{http://www.omg.org/spec/BPMN/20100524/MODEL}exclusiveGateway',
        '{http://www.omg.org/spec/BPMN/20100524/MODEL}userTask',
        '{http://www.omg.org/spec/BPMN/20100524/MODEL}serviceTask'
    ]

    # Add nodes
    for tag in bpmn_node_tags:
        for elem in root.iter(tag):
            graph.add_node(elem.attrib['id'], type=elem.tag.split('}')[-1], name=elem.attrib.get('name', ''))

    # Add edges
    for flow in root.iter('{http://www.omg.org/spec/BPMN/20100524/MODEL}sequenceFlow'):
        source_ref = flow.attrib.get('sourceRef')
        target_ref = flow.attrib.get('targetRef')
        if source_ref and target_ref:
            graph.add_edge(source_ref, target_ref)
    return graph


def validar_json_sintaticamente(json_string):
    """
    Tenta parsear a string JSON para validar sua sintaxe.
    Retorna True se o JSON for sintaticamente válido, False caso contrário.
    """
    try:
        json.loads(json_string)
        return True
    except json.JSONDecodeError:
        return False

def validar_bpmn_logicamente(process_data):
    """
    Valida a lógica BPMN de um dicionário de processo, verificando se os IDs de origem e destino
    dos sequence_flows existem na lista de elementos.
    Retorna True se for logicamente válido, False caso contrário.
    """
    elements_ids = {elem["id"] for elem in process_data.get("elements", [])}

    for flow_data in process_data.get("sequence_flows", []):
        source_id = flow_data.get("from")
        target_id = flow_data.get("to")

        if source_id not in elements_ids:
            print(f"❌ Erro de validação lógica BPMN: Fluxo '{flow_data.get('id', 'N/A')}' aponta para origem inexistente: '{source_id}'")
            return False
        if target_id not in elements_ids:
            print(f"❌ Erro de validação lógica BPMN: Fluxo '{flow_data.get('id', 'N/A')}' aponta para destino inexistente: '{target_id}'")
            return False
    return True



def run_model_until_valid_json(
    model, # Gemini model object
    prompt_sistema, # System prompt for Gemini
    sua_nova_descricao, # User description for Gemini
    generation_config, # Generation config for Gemini
    max_attempts=5
):

    retry_count = 0
    elapsed_generation_time = 0 # Initialize generation time
    while retry_count < max_attempts:
        retry_count += 1
        print(f"\n⚡ Tentativa {retry_count} de {max_attempts}: Gerando texto com Gemini...")

        # 1. Gerar o texto usando a API do Gemini
        try:
            start_generate_time = time.time() # Record start time for generation
            response = model.generate_content(
                [prompt_sistema, f"Converta este processo:\n{sua_nova_descricao}"],
                generation_config=generation_config
            )
            end_generate_time = time.time() # Record end time for generation
            elapsed_generation_time = end_generate_time - start_generate_time # Calculate elapsed time
            cleaned_output = response.text.strip()
        except Exception as e:
            print(f"❌ Erro ao chamar a API do Gemini na tentativa {retry_count}: {e}")
            # Se a API falhar, não há JSON para validar, então continuamos para a próxima tentativa
            continue

        # 2. Extrair o bloco JSON da string gerada (lógica existente)
        codigo_json = ""
        json_start_marker = "```json"
        json_end_marker = "```"

        start_idx = cleaned_output.find(json_start_marker)
        end_idx = cleaned_output.find(json_end_marker, start_idx + len(json_start_marker))

        if start_idx != -1 and end_idx != -1:
            codigo_json = cleaned_output[start_idx + len(json_start_marker):end_idx].strip()
        else:
            print("Warning: Bloco Markdown JSON não encontrado. Tentando chaves estruturais.")
            first_brace = cleaned_output.find('{')
            last_brace = cleaned_output.rfind('}')
            if first_brace != -1 and last_brace != -1 and last_brace > first_brace:
                codigo_json = cleaned_output[first_brace:last_brace+1].strip()

        # 3. Validar a sintaxe JSON (lógica existente)
        if validar_json_sintaticamente(codigo_json):
            print(f"✅ JSON sintaticamente válido encontrado na tentativa {retry_count}. Tentando validação lógica...")
            try:
                dados_processo = json.loads(codigo_json)

                # NOVA VALIDAÇÃO: Validação lógica do BPMN (consistência de flows e elementos) (lógica existente)
                if validar_bpmn_logicamente(dados_processo):
                    print(f"✅ JSON sintaticamente e logicamente válido encontrado na tentativa {retry_count}. Retornando.\n")
                    return dados_processo, retry_count, elapsed_generation_time # Return generation time
                else:
                    print(f"❌ JSON sintaticamente válido, mas falhou na validação lógica BPMN na tentativa {retry_count}. Retentando...")
            except json.JSONDecodeError as e:
                print(f"Erro inesperado ao parsear JSON válido: {e}")
        else:
            print(f"❌ JSON inválido na tentativa {retry_count}. Conteúdo bruto: {cleaned_output[:500]}...")

    print(f"\n⛔ Não foi possível gerar um JSON sintaticamente e logicamente válido após {max_attempts} tentativas.")
    return None, max_attempts, elapsed_generation_time # Return generation time even if failed

In [ ]:
# 4. Chamada de API com Gemini
# A API do Gemini não tem um parâmetro 'response_format' direto para JSON no método generate_content,
# então instruímos o modelo no prompt a retornar APENAS JSON.
json_output, num_attempts, generation_time = run_model_until_valid_json(
    model=model,
    prompt_sistema=prompt_sistema,
    sua_nova_descricao=sua_nova_descricao,
    generation_config=genai.types.GenerationConfig(
        temperature=0.1,
        max_output_tokens=8192 # Definição max_output_tokens dentro de GenerationConfig
    ),
    max_attempts=5
)

# 5. Extração e Validação do JSON
# A função `run_model_until_valid_json` já retorna o dicionário JSON validado (ou None).

if json_output is not None:
    print("\n✅ JSON extraído e validado com sucesso via Gemini:\n")
    print(json.dumps(json_output, indent=4, ensure_ascii=False))
else:
    print(f"Erro catastrófico: A API não retornou um JSON válido após {num_attempts} tentativas.")
    # Não há cleaned_output para imprimir aqui, pois a função indicou falha na geração de JSON válido.


⚡ Tentativa 1 de 5: Gerando texto com Gemini...
✅ JSON sintaticamente válido encontrado na tentativa 1. Tentando validação lógica...
✅ JSON sintaticamente e logicamente válido encontrado na tentativa 1. Retornando.


✅ JSON extraído e validado com sucesso via Gemini:

{
    "process_name": "Client_Acquisition_Process",
    "elements": [
        {
            "id": "start",
            "type": "startEvent",
            "name": "New clients needed"
        },
        {
            "id": "perform_analysis",
            "type": "userTask",
            "name": "Perform market analysis"
        },
        {
            "id": "prepare_portfolio",
            "type": "userTask",
            "name": "Prepare portfolio presentation"
        },
        {
            "id": "send_offers_inform",
            "type": "userTask",
            "name": "Send offers and inform management"
        },
        {
            "id": "send_detailed_offer",
            "type": "userTask",
            "name": "Se

In [ ]:
import xml.etree.ElementTree as ET_lxml
import spacy
from sentence_transformers import SentenceTransformer, util
import networkx as nx

# Carregando os dados de referência e o XML gerado
bpmn_xml_string = gerar_bpmn_definitivo(json_output)

# XML gerado pelo modelo (já disponível na variável bpmn_xml_string)
xml_gerado = bpmn_xml_string

# XML de referência do dataset (primeiro registro)
xml_referencia = coherence_df.loc[0, 'bpmn_xml']

# Descrição original do processo (para coerência de requisitos)
texto_referencia = sua_nova_descricao

print("Dados de referência e XML gerado carregados para as métricas.")

# Executa todas as métricas modularizadas

# Certifica-se de que nlp e model_st estão carregados
nlp = spacy.load('en_core_web_sm') # Carrega o modelo spaCy
model_st = SentenceTransformer('paraphrase-MiniLM-L6-v2') # Carrega o SentenceTransformer

# 2. Cobertura de Requisitos (Semantic Similarity)
semantic_similarity_score = calculate_requirements_coverage_metric(texto_referencia, xml_gerado, nlp, model_st)
print(f"Cobertura de Requisitos (Similaridade Semântica): {semantic_similarity_score:.2f}%")

# 3. Complexidade Estrutural (Graph Analysis)
structural_complexity_score = calculate_structural_complexity_metric(xml_gerado)
print(f"Complexidade Estrutural (Densidade do Grafo): {structural_complexity_score:.2f}%")

# 4. Graph Edit Distance (GED)
ged_similarity_score = calculate_ged_metric(xml_gerado, xml_referencia)
print(f"Graph Edit Distance (Similaridade GED Heurística): {ged_similarity_score:.2f}%")

DEBUG (_initialize_bpmn_maps): Conteúdo de process_data['elements'] no início da função: [
  {
    "id": "start",
    "type": "startEvent",
    "name": "New clients needed"
  },
  {
    "id": "perform_analysis",
    "type": "userTask",
    "name": "Perform market analysis"
  },
  {
    "id": "prepare_portfolio",
    "type": "userTask",
    "name": "Prepare portfolio presentation"
  },
  {
    "id": "send_offers_inform",
    "type": "userTask",
    "name": "Send offers and inform management"
  },
  {
    "id": "send_detailed_offer",
    "type": "userTask",
    "name": "Send detailed offer to client"
  },
  {
    "id": "gateway_accepted",
    "type": "exclusiveGateway",
    "name": "Offer accepted?"
  },
  {
    "id": "prepare_sign_contract",
    "type": "userTask",
    "name": "Prepare and sign contract"
  },
  {
    "id": "analyze_failure",
    "type": "userTask",
    "name": "Analyze failure reason"
  },
  {
    "id": "end",
    "type": "endEvent",
    "name": "Process End"
  }
]
DEBU

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Cobertura de Requisitos (Similaridade Semântica): 74.99%
Complexidade Estrutural (Densidade do Grafo): 12.50%
Graph Edit Distance (Similaridade GED Heurística): 64.29%


In [ ]:
import pandas as pd

# Calcula a Taxa de Erro Sintático com base nas tentativas de geração de JSON
# 'attempts' é o número de tentativas até um JSON válido ser encontrado (ou max_attempts se falhar)
# 'json_output' é None se nenhum JSON válido foi gerado

if json_output: # Um JSON válido foi encontrado
    failed_attempts = num_attempts - 1
    total_attempts_made = num_attempts
    if total_attempts_made > 0:
        taxa_erro_sintatico = (failed_attempts / total_attempts_made) * 100
    else: # Caso teoricamente não alcançado se attempts > 0
        taxa_erro_sintatico = 0.0
else: # Nenhum JSON válido foi encontrado após max_attempts
    taxa_erro_sintatico = 100.0 # Todos os attempts falharam

# Coleta os valores das métricas
metrics_data = {
    'Métrica': [
        'Taxa de Erro Sintático (JSON Gerado)', # Renomeado para maior clareza
        'Cobertura de Requisitos (Similaridade Semântica)',
        'Complexidade Estrutural (Densidade do Grafo)',
        'Graph Edit Distance (Similaridade GED Heurística)',
        'Tempo de Execução (Geração JSON)' # Adicionado a nova métrica
    ],
    'Valor': [
        f'{taxa_erro_sintatico:.2f}%',
        f'{semantic_similarity_score:.2f}%',
        f'{structural_complexity_score:.2f}%',
        f'{ged_similarity_score:.2f}%',
        f'{generation_time:.2f} segundos' # Adicionado o valor da nova métrica
    ]
}

# Cria o DataFrame e exibe
df_metrics = pd.DataFrame(metrics_data)
display(df_metrics)

,Métrica,Valor
0,Taxa de Erro Sintático (JSON Gerado),0.00%
1,Cobertura de Requisitos (Similaridade Semântica),74.99%
2,Complexidade Estrutural (Densidade do Grafo),12.50%
3,Graph Edit Distance (Similaridade GED Heurística),64.29%
4,Tempo de Execução (Geração JSON),9.94 segundos
